## 第 1 课：环境搭建与 Raw Generation 基线

### 本课目标

- 配好项目 Python 环境
- 安装并验证 Ollama
- 跑通一次 raw generation
- 建立项目 notebook 的最小可运行骨架

In [1]:
import ollama

model_name="qwen3.5:4b"

def generate_raw_response(prompt,model=model_name,temperature=0.1,num_predict=200):
    response = ollama.generate(
        model=model,
        prompt=prompt,
        raw=True,
        options={
            "temperature": temperature,
            "num_predict": num_predict
        }
    )
    return {
        "text": response["response"],
        "prompt_tokens": response.get("prompt_eval_count"),
        "output_tokens": response.get("eval_count"),
        "done": response.get("done"),
    }

In [ ]:
import json
test_prompt = "Answer in one short sentence only.\nQuestion: What does HKBU stand for?\nAnswer:"
result=generate_raw_response(test_prompt, temperature=0.1, num_predict=500)
print("==== 完整返回信息 ====")
print(json.dumps(result, indent=4, ensure_ascii=False))
# 提取纯文本
print("\n==== 纯文本回答 ====")
print(result["text"])


==== 完整返回信息 ====
{
    "text": "\n\n<think>\nThinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: \"Answer in one short sentence only. Question: What does HKBU stand for?\"\n    *   Constraint: \"one short sentence only\".\n    *   Task: Identify the meaning of the acronym \"HKBU\".\n\n2.  **Identify the Acronym:**\n    *   \"HKBU\" is a common abbreviation for a specific institution.\n    *   It stands for \"Hong Kong Baptist University\".\n\n3.  **Draft the Answer:**\n    *   Draft 1: HKBU stands for Hong Kong Baptist University.\n    *   Draft 2: It stands for Hong Kong Baptist University.\n    *   Draft 3: HKBU is the abbreviation for Hong Kong Baptist University.\n\n4.  **Check Constraints:**\n    *   \"one short sentence only\": Draft 1 fits perfectly.\n    *   \"Answer in one short sentence only\": The prompt asks me to answer the question, but also imposes a formatting constraint on the answer itself.\n\n5.  **Refine for Accuracy and Brevity:**\n    *   \"HKBU stan

## 第 2 课：理解模型局限、Token 与生成参数

### 本课目标

- 理解 hallucination、counting mistakes、tokenization 的意义
- 知道为什么 token efficiency 是评分点
- 设计你项目中的 generation parameter 实验方案

In [12]:
import pandas as pd 
#from scripts.generation import generate_raw_response
from tqdm import tqdm

questions = [
    "What is Prompt Engineering?",
    "What is Chain of Thought prompting?",
    "What is the difference between few-shot prompting and zero-shot prompting?",
]
temperatures = [0.0, 0.3, 0.7]
num_predicts = [100, 500, 1000]

rows = []

total_iters = len(temperatures) * len(num_predicts) * len(questions)

with tqdm(total=total_iters, desc="testing with different parameters") as pbar:
    for temperature in temperatures:
        for num_predict in num_predicts:
            for question in questions:
                result = generate_raw_response(
                    prompt=f"Answer briefly in one short sentence only.\n\n{question}",
                    temperature=temperature,
                    num_predict=num_predict
                )
                
                rows.append({
                    "query": question,
                    "temperature": temperature,
                    "num_predict": num_predict,
                    "response": result["text"],
                    "prompt_tokens": result["prompt_tokens"],
                    "output_tokens": result["output_tokens"],
                })
                
                pbar.update(1)

df = pd.DataFrame(rows)
df["quality_score"] = None
df["notes"] = ""
df.head()

testing with different parameters: 100%|██████████| 27/27 [11:18<00:00, 25.14s/it]


,query,temperature,num_predict,response,prompt_tokens,output_tokens,quality_score,notes
0,What is Prompt Engineering?,0.0,100,\n\n<think>\nThinking Process:\n\n1. **Analyz...,14,100,None,
1,What is Chain of Thought prompting?,0.0,100,\n\n<think>\nThinking Process:\n\n1. **Analyz...,16,100,None,
2,What is the difference between few-shot prompt...,0.0,100,\n\n<think>\nThinking Process:\n\n1. **Analyz...,22,100,None,
3,What is Prompt Engineering?,0.0,500,\n\n<think>\nThinking Process:\n\n1. **Analyz...,14,500,None,
4,What is Chain of Thought prompting?,0.0,500,\n\n<think>\nThinking Process:\n\n1. **Analyz...,16,500,None,


In [13]:
from pathlib import Path

Path("results").mkdir(exist_ok=True)
df.to_csv("results/lesson2_parameter_sweep.csv", index=False)


## 第 3 课：多轮对话、History 与 System Message

### 本课目标

- 做出最小可聊天的课程助手
- 支持 conversation history
- 支持 system message 控制回答身份和边界

In [5]:
from scripts.memory import ConversationMemory
from scripts.generation import generate_raw_response

#model_name="qwen3.5:4b"
model_name="llama3.1:8b"
system_prompt = "You are an HKBU Study Companion. Answer briefly and clearly."

memory = ConversationMemory(system_prompt=system_prompt, max_turns=3)

user_message = "What does HKBU stand for?"

prompt = memory.build_chat_prompt(user_message)

result = generate_raw_response(
    model=model_name,
    prompt=prompt,
    temperature=0.1,
    num_predict=120
)

assistant_message = result["text"].strip()

print("Prompt:")
print(prompt)
print(assistant_message)

memory.add_user_message(user_message)
memory.add_assistant_message(assistant_message)


Prompt:
You are an HKBU Study Companion. Answer briefly and clearly.

Conversation Hisory:
(empty)

Current User Message:
What does HKBU stand for?

Assistant:
HKBU stands for Hong Kong Baptist University. 

Would you like to know more about the university?  Please feel free to ask!


In [6]:
user_message = "Can you explain it in Chinese?"

prompt = memory.build_chat_prompt(user_message)

result = generate_raw_response(
    model=model_name,
    prompt=prompt,
    temperature=0.1,
    num_predict=120
)

assistant_message = result["text"].strip()

print("Prompt:")
print(prompt)
print(assistant_message)

memory.add_user_message(user_message)
memory.add_assistant_message(assistant_message)


Prompt:
You are an HKBU Study Companion. Answer briefly and clearly.

Conversation Hisory:
User: What does HKBU stand for?
Assistant: HKBU stands for Hong Kong Baptist University. 

Would you like to know more about the university?  Please feel free to ask!

Current User Message:
Can you explain it in Chinese?

Assistant:
HKBU 的中文全稱是香港浸會大學。 

Do you want to learn more about the university's history or its faculties?  I'm here to help!


In [ ]:
from scripts.memory import ConversationMemory
from scripts.generation import generate_raw_response

user_message = "What does HKBU stand for?"

prompt_a = "You are an HKBU Study Companion. Answer in one short sentence only. Do not ask follow-up questions."
prompt_b = "You are an HKBU Study Companion. Answer in Chinese."

memory_a = ConversationMemory(system_prompt=prompt_a, max_turns=3)
memory_b = ConversationMemory(system_prompt=prompt_b, max_turns=3)

result_a = generate_raw_response(
    prompt=memory_a.build_chat_prompt(user_message),
    model="llama3.1:8b",
    temperature=0.1,
    num_predict=80
)

result_b = generate_raw_response(
    prompt=memory_b.build_chat_prompt(user_message),
    model="llama3.1:8b",
    temperature=0.1,
    num_predict=80
)

print("=== Prompt A ===")
print(result_a["text"])
print("\n=== Prompt B ===")
print(result_b["text"])


=== Prompt A ===
 Hong Kong Baptist University.  Created by HKBU Study Companion.  How can I assist you today?

=== Prompt B ===
 
HKBU stands for Hong Kong Baptist University. 

Extra Explanation: HKBU is a public university located in Kowloon Tong, Hong Kong. It was established in 1956 as the Hong Kong Baptist College and became a full-fledged university in 1994. HKBU is known for its strong programs in business, education, humanities, social sciences, and science. 

What's your
